# 第7章：向量数据库

> 🎯 掌握Embedding和ChromaDB向量存储

In [ ]:
# 安装依赖
# !pip install chromadb sentence-transformers

## 1. Embedding 文本向量化

In [ ]:
from langchain_community.embeddings import HuggingFaceEmbeddings

embeddings = HuggingFaceEmbeddings(
    model_name='sentence-transformers/all-MiniLM-L6-v2'
)

text = 'Python是一门编程语言'
vector = embeddings.embed_query(text)
print(f'文本: {text}')
print(f'向量维度: {len(vector)}')
print(f'向量前5维: {vector[:5]}')

In [ ]:
# 相似度计算
import numpy as np

def cosine_similarity(a, b):
    return np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b))

texts = ['Python编程', 'Java开发', '今天天气真好']
vectors = [embeddings.embed_query(t) for t in texts]

query = embeddings.embed_query('Python是什么？')
for t, v in zip(texts, vectors):
    sim = cosine_similarity(query, v)
    print(f'{t}: {sim:.4f}')

## 2. ChromaDB 向量存储

In [ ]:
from langchain_chroma import Chroma
from langchain_core.documents import Document

# 准备文档
docs = [
    Document(page_content='Python是解释型语言', metadata={'topic': 'python'}),
    Document(page_content='Java是编译型语言', metadata={'topic': 'java'}),
    Document(page_content='机器学习需要大量数据', metadata={'topic': 'ml'}),
]

# 创建向量库
vectorstore = Chroma.from_documents(
    documents=docs,
    embedding=embeddings,
    persist_directory='./chroma_demo'
)
print('向量库创建完成')

In [ ]:
# 相似度搜索
results = vectorstore.similarity_search('Python是什么类型的语言？', k=2)
for doc in results:
    print(f'- {doc.page_content} [{doc.metadata}]')

In [ ]:
# 带分数的搜索
results = vectorstore.similarity_search_with_score('编程语言', k=3)
for doc, score in results:
    print(f'{score:.4f} - {doc.page_content}')

## 3. 检索器

In [ ]:
# 转为检索器
retriever = vectorstore.as_retriever(search_kwargs={'k': 2})
docs = retriever.invoke('Python编程')
for doc in docs:
    print(f'- {doc.page_content}')

## 📝 小结
1. **Embedding**：文本转向量
2. **ChromaDB**：向量存储与检索
3. **Retriever**：检索器接口